# Certainty Labs -- EORM Training on Kaggle T4 GPU

Trains a TransEBM energy model on GSM8K-Llama3 Chain-of-Thought data using the full EORM architecture.

- **Data**: `/kaggle/input/certainty-eorm-data/` (uploaded via Kaggle Datasets API)
- **Output**: `/kaggle/working/` (model .pt, tokenizer, metrics)
- **GPU**: T4 16GB with FP16 mixed precision
- **Architecture**: Matches [EnergyORM](https://github.com/ericjiang18/EnergyORM)

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate pydantic pyyaml jsonlines tqdm plotly

In [ ]:
import torch
import json
import os
import sys
import time
import copy
from collections import defaultdict
from functools import partial
from contextlib import nullcontext

print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', 0) / 1e9
    print(f'GPU: {gpu} ({mem:.1f} GB)')
else:
    print('WARNING: No GPU detected')

## Locate Training Data

In [ ]:
# Find training data -- recursively search /kaggle/input and local demo_dataset
import glob

KAGGLE_BASE = '/kaggle/input'
LOCAL_DIR = 'demo_dataset'

TRAIN_FILE = None
TEST_FILE = None

# Walk the entire /kaggle/input tree to find .jsonl files
if os.path.isdir(KAGGLE_BASE):
    print('Scanning /kaggle/input/ ...')
    for root, dirs, files in os.walk(KAGGLE_BASE):
        for f in files:
            print(f'  Found: {os.path.join(root, f)}')
            if f.endswith('.jsonl'):
                path = os.path.join(root, f)
                if 'train' in f and TRAIN_FILE is None:
                    TRAIN_FILE = path
                elif 'test' in f and TEST_FILE is None:
                    TEST_FILE = path

# Fallback to local demo_dataset
if TRAIN_FILE is None and os.path.isdir(LOCAL_DIR):
    print('Scanning local demo_dataset/ ...')
    for f in os.listdir(LOCAL_DIR):
        if f.endswith('.jsonl'):
            path = os.path.join(LOCAL_DIR, f)
            if 'train' in f and TRAIN_FILE is None:
                TRAIN_FILE = path
            elif 'test' in f and TEST_FILE is None:
                TEST_FILE = path

assert TRAIN_FILE, f'Training data not found under {KAGGLE_BASE} or {LOCAL_DIR}'
print(f'Train: {TRAIN_FILE}')
print(f'Test:  {TEST_FILE}')

train_lines = sum(1 for _ in open(TRAIN_FILE))
print(f'Train lines: {train_lines:,}')

## Define Model Architecture (TransEBM)

In [ ]:
import torch.nn as nn

class TransEBM(nn.Module):
    def __init__(self, vocab_size, d_model=768, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.d_model = d_model
        self.emb = nn.Embedding(vocab_size, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model, n_heads, dim_feedforward=4*d_model,
            activation='gelu', dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.enc = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
        )

    def forward(self, ids, mask):
        x = self.emb(ids) * (self.d_model ** 0.5)
        padding_mask = (mask == 0)
        x = self.enc(x, src_key_padding_mask=padding_mask)
        cls_repr = x[:, 0]
        return self.head(cls_repr).squeeze(-1)

    def resize_token_embeddings(self, new_num_tokens):
        old_emb = self.emb
        if old_emb.num_embeddings == new_num_tokens:
            return
        new_emb = nn.Embedding(new_num_tokens, self.d_model)
        new_emb.weight.data.normal_(mean=0.0, std=0.02)
        n_copy = min(old_emb.num_embeddings, new_num_tokens)
        new_emb.weight.data[:n_copy, :] = old_emb.weight.data[:n_copy, :]
        self.emb = new_emb

print('TransEBM defined')

## Utilities: Tokenizer, Loss, Data Loading

In [ ]:
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from tqdm import tqdm


def setup_tokenizer(name='gpt2', max_length=2048):
    tok = AutoTokenizer.from_pretrained(name)
    tok.model_max_length = max_length
    if tok.pad_token_id is None:
        if tok.eos_token_id is not None:
            tok.pad_token = tok.eos_token
        else:
            tok.add_special_tokens({'pad_token': '[PAD]'})
    pad_id = tok.pad_token_id
    cls_id = tok.bos_token_id or tok.cls_token_id or tok.eos_token_id
    return tok, pad_id, cls_id


def bradley_terry_loss(e, l):
    pos_idx = torch.where(l == 1)[0]
    neg_idx = torch.where(l == 0)[0]
    if len(pos_idx) == 0 or len(neg_idx) == 0:
        return None
    diffs = e[pos_idx].unsqueeze(1) - e[neg_idx].unsqueeze(0)
    return F.softplus(diffs).mean()


def load_q2cands(path):
    q2c = defaultdict(list)
    with open(path, encoding='utf-8') as f:
        for line in f:
            try:
                ex = json.loads(line)
                if 'question' in ex and 'label' in ex and ('gen_text' in ex or 'generated_full_text' in ex):
                    q2c[ex['question']].append(ex)
            except json.JSONDecodeError:
                continue
    print(f'Loaded {sum(len(v) for v in q2c.values())} candidates across {len(q2c)} questions')
    return dict(q2c)


print('Utilities defined')

## Dataset Classes

In [ ]:
import random

class TrainValChunkDS(Dataset):
    def __init__(self, tokenizer, max_length, cls_id, pad_id, q2cands_data, split='train', holdout=0.2):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.cls_id = cls_id
        self.pad_id = pad_id
        self.groups = []

        # Filter for diversity
        filtered = {}
        for q, cands in q2cands_data.items():
            has_pos = any(e['label'] == 1 for e in cands)
            has_neg = any(e['label'] == 0 for e in cands)
            if has_pos and has_neg:
                filtered[q] = cands
        print(f'Diversity filter: {len(filtered)}/{len(q2cands_data)} groups kept')

        # Tokenize
        sep = tokenizer.eos_token or '\n'
        for q, cands in tqdm(filtered.items(), desc=f'Tokenizing ({split})'):
            enc_grp = []
            has_correct = False
            for e in cands:
                if e['label'] == 1:
                    has_correct = True
                answer = e.get('gen_text') or e.get('generated_full_text')
                if answer is None:
                    continue
                ids = tokenizer.encode(
                    f'{q}{sep}{answer}',
                    add_special_tokens=False,
                    truncation=True,
                    max_length=max_length - 1,
                )
                enc_grp.append({
                    'ids': torch.tensor([cls_id] + ids, dtype=torch.long),
                    'lab': torch.tensor(e['label'], dtype=torch.float),
                })
            if enc_grp:
                self.groups.append({'candidates': enc_grp, 'meta': {'has_correct': has_correct}})

        # Split
        random.seed(42)
        random.shuffle(self.groups)
        cut = int((1.0 - holdout) * len(self.groups))
        if split == 'train':
            self.groups = self.groups[:cut]
        else:
            self.groups = self.groups[cut:]
        print(f'{split} split: {len(self.groups)} groups')

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        return self.groups[idx]['candidates']


def collate_fn(batch, pad_id):
    idsL, maskL, labL = [], [], []
    for grp in batch:
        if not grp:
            continue
        valid = [e for e in grp if 'ids' in e and 'lab' in e]
        if not valid:
            continue
        ids = [e['ids'].cpu() for e in valid]
        labs = [e['lab'] for e in valid]
        padded = pad_sequence(ids, batch_first=True, padding_value=pad_id)
        mask = (padded != pad_id).long()
        idsL.append(padded)
        maskL.append(mask)
        labL.append(torch.stack(labs))
    return idsL, maskL, labL


print('Dataset classes defined')

## Configuration

In [ ]:
# Hyperparameters (matching EnergyORM defaults)
CFG = {
    'tokenizer': 'gpt2',
    'd_model': 768,
    'n_heads': 4,
    'n_layers': 2,
    'dropout': 0.2,
    'max_length': 2048,
    'epochs': 20,
    'batch_size': 1,
    'lr': 5e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'val_holdout': 0.2,
    'validate_every': 1,
    'save_prefix': 'ebm_certainty',
}

OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else 'certainty_workspace/model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Output directory: {OUTPUT_DIR}')
print(f'Config: {json.dumps(CFG, indent=2)}')

## Load Data and Prepare DataLoaders

In [ ]:
# Setup
tok, PAD_ID, CLS_ID = setup_tokenizer(CFG['tokenizer'], CFG['max_length'])

# Device and AMP
if torch.cuda.is_available():
    DEV = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEV = torch.device('mps')
else:
    DEV = torch.device('cpu')

use_amp = DEV.type == 'cuda'
autocaster = torch.cuda.amp.autocast if use_amp else nullcontext
scaler = torch.cuda.amp.GradScaler() if use_amp else None

print(f'Device: {DEV}, FP16 AMP: {use_amp}')
print(f'Tokenizer vocab: {len(tok)}, PAD_ID: {PAD_ID}, CLS_ID: {CLS_ID}')

In [ ]:
# Load data
q2cands = load_q2cands(TRAIN_FILE)

# Create train/val datasets
train_ds = TrainValChunkDS(tok, CFG['max_length'], CLS_ID, PAD_ID,
                           q2cands_data=copy.deepcopy(q2cands), split='train', holdout=CFG['val_holdout'])
val_ds = TrainValChunkDS(tok, CFG['max_length'], CLS_ID, PAD_ID,
                         q2cands_data=copy.deepcopy(q2cands), split='val', holdout=CFG['val_holdout'])

# DataLoaders
collate_with_pad = partial(collate_fn, pad_id=PAD_ID)
pin_mem = DEV.type == 'cuda'

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                      collate_fn=collate_with_pad, pin_memory=pin_mem, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                    collate_fn=collate_with_pad, pin_memory=pin_mem, num_workers=2)

# Oracle accuracy
oracle_hits = sum(1 for g in val_ds.groups if g['meta'].get('has_correct', False))
print(f'Oracle val accuracy: {100*oracle_hits/len(val_ds.groups):.1f}% ({oracle_hits}/{len(val_ds.groups)})')

## Train

In [ ]:
# Initialize model
model = TransEBM(
    vocab_size=len(tok),
    d_model=CFG['d_model'],
    n_heads=CFG['n_heads'],
    n_layers=CFG['n_layers'],
    dropout=CFG['dropout'],
).to(DEV)

if len(tok) != model.emb.num_embeddings:
    model.resize_token_embeddings(len(tok))

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params/1e6:.1f}M')

# Optimizer and scheduler
opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
total_steps = CFG['epochs'] * len(train_dl)
warmup_steps = int(CFG['warmup_ratio'] * total_steps)
sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

print(f'Total steps: {total_steps}, Warmup: {warmup_steps}')

In [ ]:
# Training loop (matching EnergyORM step1_train_ebm.py)
best_val_acc = 0.0
best_state = None
history = []
start_time = time.time()

for ep in range(1, CFG['epochs'] + 1):
    model.train()
    total_loss = 0.0
    n_batches = 0

    pbar = tqdm(train_dl, desc=f'Epoch {ep}/{CFG["epochs"]}', unit='batch')
    for idsL, maskL, labL in pbar:
        opt.zero_grad(set_to_none=True)
        batch_losses = []

        for ids_b, mask_b, lab_b in zip(idsL, maskL, labL):
            if ids_b.numel() == 0:
                continue
            ids_b, mask_b, lab_b = ids_b.to(DEV), mask_b.to(DEV), lab_b.to(DEV)
            with autocaster():
                e = model(ids_b, mask_b)
                loss_grp = bradley_terry_loss(e, lab_b)
            if loss_grp is not None and torch.isfinite(loss_grp):
                batch_losses.append(loss_grp)

        if not batch_losses:
            continue

        loss = torch.stack(batch_losses).mean()

        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        sched.step()
        total_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{sched.get_last_lr()[0]:.2e}')

    avg_loss = total_loss / max(n_batches, 1)

    # Validation
    val_acc = 0.0
    naive_acc = 0.0
    if ep % CFG['validate_every'] == 0 and val_dl is not None:
        model.eval()
        total_groups = 0
        ebm_correct = 0
        naive_correct = 0
        with torch.no_grad():
            for idsL, maskL, labL in val_dl:
                for ids, mask, lab in zip(idsL, maskL, labL):
                    if ids.numel() == 0:
                        continue
                    ids, mask, lab = ids.to(DEV), mask.to(DEV), lab.to(DEV)
                    with autocaster():
                        e = model(ids, mask)
                    best_idx = torch.argmin(e)
                    if lab[best_idx].item() == 1:
                        ebm_correct += 1
                    if lab[0].item() == 1:
                        naive_correct += 1
                    total_groups += 1
        if total_groups > 0:
            val_acc = 100.0 * ebm_correct / total_groups
            naive_acc = 100.0 * naive_correct / total_groups

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            print(f'  >> New best: {best_val_acc:.2f}%')

    history.append({'epoch': ep, 'loss': avg_loss, 'val_acc': val_acc})
    print(f'Epoch {ep} | Loss: {avg_loss:.4f} | Val EBM: {val_acc:.2f}% | Val Naive: {naive_acc:.2f}%')

elapsed = time.time() - start_time
print(f'\nTraining complete in {elapsed:.0f}s. Best val acc: {best_val_acc:.2f}%')

## Save Model and Metrics

In [ ]:
# Save best model
model_path = os.path.join(OUTPUT_DIR, f'{CFG["save_prefix"]}_model.pt')
tokenizer_path = os.path.join(OUTPUT_DIR, f'{CFG["save_prefix"]}_tokenizer')

if best_state is not None:
    model.load_state_dict(best_state)

save_data = {
    'state_dict': model.state_dict(),
    'd_model': CFG['d_model'],
    'vocab_size': model.emb.num_embeddings,
    'n_layers': CFG['n_layers'],
    'n_heads': CFG['n_heads'],
}
torch.save(save_data, model_path)
print(f'Model saved: {model_path}')

os.makedirs(tokenizer_path, exist_ok=True)
tok.save_pretrained(tokenizer_path)
print(f'Tokenizer saved: {tokenizer_path}')

# Save metrics
metrics = {
    'best_val_acc': best_val_acc,
    'final_loss': history[-1]['loss'] if history else 0,
    'epochs_trained': len(history),
    'elapsed_seconds': round(elapsed, 1),
    'model_path': model_path,
    'tokenizer_path': tokenizer_path,
    'config': CFG,
    'history': history,
}
metrics_path = os.path.join(OUTPUT_DIR, f'{CFG["save_prefix"]}_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Metrics saved: {metrics_path}')
print(f'\n{json.dumps({k: v for k, v in metrics.items() if k != "history"}, indent=2)}')

## Training Curves

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=('Training Loss', 'Validation Accuracy'))

fig.add_trace(
    go.Scatter(y=[h['loss'] for h in history], mode='lines+markers',
               line=dict(color='#6C5CE7', width=2)),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(y=[h['val_acc'] for h in history], mode='lines+markers',
               line=dict(color='#00b894', width=2)),
    row=1, col=2,
)
fig.update_layout(height=350, showlegend=False, template='plotly_dark')
fig.update_xaxes(title_text='Epoch')
fig.show()

## Test Set Evaluation

In [ ]:
if TEST_FILE:
    print(f'Evaluating on test set: {TEST_FILE}')
    test_q2c = load_q2cands(TEST_FILE)

    # Tokenize test data (no diversity filter, no split)
    sep = tok.eos_token or '\n'
    test_groups = []
    for q, cands in tqdm(test_q2c.items(), desc='Tokenizing test'):
        enc_grp = []
        for e in cands:
            answer = e.get('gen_text') or e.get('generated_full_text')
            if answer is None:
                continue
            ids = tok.encode(f'{q}{sep}{answer}', add_special_tokens=False,
                            truncation=True, max_length=CFG['max_length'] - 1)
            enc_grp.append({
                'ids': torch.tensor([CLS_ID] + ids, dtype=torch.long),
                'lab': torch.tensor(e['label'], dtype=torch.float),
            })
        if enc_grp:
            test_groups.append(enc_grp)

    class SimpleDS(Dataset):
        def __init__(self, groups): self.groups = groups
        def __len__(self): return len(self.groups)
        def __getitem__(self, i): return self.groups[i]

    test_dl = DataLoader(SimpleDS(test_groups), batch_size=1, shuffle=False,
                         collate_fn=collate_with_pad, num_workers=2)

    model.eval()
    total = 0
    ebm_ok = 0
    naive_ok = 0
    with torch.no_grad():
        for idsL, maskL, labL in tqdm(test_dl, desc='Testing'):
            for ids, mask, lab in zip(idsL, maskL, labL):
                if ids.numel() == 0:
                    continue
                ids, mask, lab = ids.to(DEV), mask.to(DEV), lab.to(DEV)
                with autocaster():
                    e = model(ids, mask)
                if lab[torch.argmin(e)].item() == 1:
                    ebm_ok += 1
                if lab[0].item() == 1:
                    naive_ok += 1
                total += 1

    test_ebm_acc = 100 * ebm_ok / max(total, 1)
    test_naive_acc = 100 * naive_ok / max(total, 1)
    print(f'\nTest Results:')
    print(f'  EBM Accuracy:   {test_ebm_acc:.2f}%')
    print(f'  Naive Accuracy: {test_naive_acc:.2f}%')
    print(f'  Total groups:   {total}')
else:
    print('No test file found, skipping test evaluation')

In [ ]:
# List output files
print('\nOutput files:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(path):
        size = os.path.getsize(path)
        print(f'  {f} ({size/1e6:.1f} MB)' if size > 1e6 else f'  {f} ({size/1e3:.1f} KB)')
    elif os.path.isdir(path):
        print(f'  {f}/ (directory)')

print('\nDone! Download output files from the Kaggle Output tab.')